# AC&CD: Active C&C Detector — Sentinel Data Lake Edition

**Adapted from:** [Cyb3r-Monk/ACCD](https://github.com/Cyb3r-Monk/ACCD)  
**Original author:** Mehmet E. (@Cyb3rMonk)  
**Adaptation:** Sentinel data lake / PySpark port

---

## What this notebook does

Detects active C2 beaconing by analysing **time delta distribution** and **data size distribution** of network traffic between the same source and destination pairs.

The key insight: rather than looking for fixed-interval beaconing (which attackers defeat by changing sleep/jitter dynamically), ACCD looks for the **active keyboard period** — the window where an attacker is issuing commands. During this phase:
- Connection intervals cluster tightly (low dispersion in the lower percentiles)
- Some connections carry larger payloads (tool output / lateral movement data)

## What changed from the original

| Area | Original | This version |
|---|---|---|
| Data source | CSV file | Sentinel data lake via `MicrosoftSentinelProvider` |
| DataFrame type | pandas | PySpark (stays distributed until final display) |
| Grouping/aggregation | `pandas groupby().agg(list)` | PySpark `groupBy()` + `collect_list()` |
| Time delta calculation | `pd.Series.diff()` inside a lambda | PySpark `Window` + `lag()` |
| Statistical functions | `numpy` percentile/median | PySpark `percentile_approx()` + pandas UDFs |
| Scoring functions | pandas `apply()` row-wise lambdas | Same logic, wrapped as PySpark UDFs |
| Final display | pandas DataFrame | `.toPandas()` called only at the end |

---

## Cell 1 — Imports

**What's happening:**  
- `MicrosoftSentinelProvider` is the VS Code extension's Python class that talks to the data lake Spark cluster. It replaces `pd.read_csv`.
- PySpark imports replace numpy/pandas equivalents for distributed operations.
- We still import pandas and numpy — they're used *inside* UDFs (functions that run on each row) and for the final display table.

In [ ]:
# Sentinel data lake provider — requires the Microsoft Sentinel VS Code extension
from sentinel_lake.providers import MicrosoftSentinelProvider

# PySpark — the distributed DataFrame API
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, LongType

# pandas + numpy — used inside UDFs and for final display only
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

# Initialise the data provider — 'spark' is the SparkSession injected by the extension
data_provider = MicrosoftSentinelProvider(spark)

print("Imports OK")

## Cell 2 — Configuration

**What's happening:**  
All tunable parameters in one place. Change `WORKSPACE_NAME` to match your Sentinel workspace.  

The column name variables map the Sentinel data lake schema to the names the algorithm uses internally. `CommonSecurityLog` is the CEF-normalised proxy/firewall table — the same data source the original notebook targeted.  

> **Note on table choice:** If your environment uses ASIM-normalised tables, swap `CommonSecurityLog` for `ASimNetworkSessionLogs` and adjust the column mappings accordingly (e.g. `SrcIpAddr` instead of `SourceIP`).

In [ ]:
# ── Workspace ────────────────────────────────────────────────────────────────
WORKSPACE_NAME = "law-sentinel-main"  # CHANGE THIS

# ── Data lake table ───────────────────────────────────────────────────────────
# CommonSecurityLog = CEF proxy/firewall logs (Squid, ZScaler, Palo Alto, etc.)
# Swap for ASimNetworkSessionLogs if your environment uses ASIM normalisation
TABLE_NAME = "CommonSecurityLog"

# ── Lookback window ───────────────────────────────────────────────────────────
# How many days of data to analyse. More = better detection of slow beacons,
# but longer Spark job. 7 days is a reasonable starting point.
LOOKBACK_DAYS = 7

# ── Column name mapping (CommonSecurityLog schema) ────────────────────────────
# If you change TABLE_NAME, update these to match the new table's schema.
f_timestamp      = 'TimeGenerated'
f_src_ip         = 'SourceIP'
f_dst_ip         = 'DestinationIP'
f_dst_host       = 'DestinationHostName'
f_dst_port       = 'DestinationPort'
f_http_method    = 'RequestMethod'
f_sent_bytes     = 'SentBytes'
f_received_bytes = 'ReceivedBytes'

# Columns to keep after loading (reduces shuffle data volume in Spark)
COLUMNS_TO_KEEP = [
    f_timestamp, f_src_ip, f_dst_ip, f_dst_host,
    f_dst_port, f_http_method, f_sent_bytes, f_received_bytes
]

# Grouping key: what defines a unique beacon "stream"
GROUP_BY_COLS = [f_src_ip, f_dst_host, f_http_method]

# ── Scoring thresholds (match the original notebook) ─────────────────────────
MIN_CONN_COUNT              = 24    # Minimum total connections to consider
MIN_INTERACTIVITY_DURATION  = 3600  # Seconds — 1 hour of active-phase connections
MIN_INTERACTIVITY_CONN      = 24    # Connections within the active phase
MIN_EXECUTION_COUNT         = 10    # Number of large-response connections expected
MIN_EXECUTION_BYTES         = 20000 # Bytes — threshold for "tool execution" response
JITTER_THRESHOLD_TS         = 55    # % — time delta jitter tolerance
JITTER_THRESHOLD_DS         = 25    # % — data size jitter tolerance
MIN_SCORE                   = 0.85  # Final score cutoff
MAX_PREVALENCE              = 5     # Max distinct source IPs to same dest (FP filter)

print("Configuration OK")

## Cell 3 — Load data from the data lake

**What's happening:**  
`data_provider.read_table()` returns a **PySpark DataFrame** — not a pandas one. The data stays distributed across the Spark cluster nodes; nothing is pulled into local memory yet.

We immediately:
1. Filter to the lookback window (pushes the time filter down to the storage layer — much cheaper than loading everything)
2. Select only the columns we need (reduces data shuffled across the network)
3. Drop rows where critical fields are null

The `.printSchema()` and `.count()` calls are diagnostic — they force a Spark action so you can verify the data looks right before proceeding.

In [ ]:
from datetime import datetime, timedelta

# Calculate the start of the lookback window
lookback_start = datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)

# Read from the data lake — this returns a PySpark DataFrame
# The Spark session connects to your workspace's managed Spark cluster
raw_df = data_provider.read_table(TABLE_NAME, WORKSPACE_NAME)

# Filter to lookback window + select required columns + drop nulls in key fields
# NOTE: F.col() is the PySpark equivalent of referencing a column by name
http_df = (
    raw_df
    .filter(F.col(f_timestamp) >= lookback_start)
    .select(COLUMNS_TO_KEEP)
    .dropna(subset=[f_timestamp, f_src_ip, f_dst_host, f_sent_bytes, f_received_bytes])
)

# Diagnostic: verify schema and row count
http_df.printSchema()
print(f"Row count (last {LOOKBACK_DAYS} days): {http_df.count():,}")

## Cell 4 — Group connections and calculate time deltas

**What's happening (and why it's different from the original):**

The original used `groupby().agg(list)` to collect all timestamps per host-pair into a Python list, then called `pd.Series.diff()` on each list to get time deltas.

In PySpark we can't do that directly because the data is distributed. Instead:
1. We use a **Window function** — think of it as "for each row, look at other rows in the same group, ordered by time"
2. `lag()` gets the *previous* timestamp in the same group
3. Subtracting gives us the time delta in seconds for each individual connection row
4. We then `groupBy()` and collect everything into arrays per host-pair using `collect_list()`

This approach scales to billions of rows — the original pandas approach would OOM.

In [ ]:
# Step 1: Define a Window — partition by our grouping key, order by time
# This is the PySpark equivalent of "for each src+dst+method pair, sort by timestamp"
w = Window.partitionBy(GROUP_BY_COLS).orderBy(f_timestamp)

# Step 2: Calculate per-row time delta in seconds using lag()
# lag() looks at the previous row in the same Window partition
# unix_timestamp() converts the timestamp to seconds since epoch for arithmetic
http_df = http_df.withColumn(
    'delta_seconds',
    F.unix_timestamp(F.col(f_timestamp)) - F.unix_timestamp(F.lag(f_timestamp, 1).over(w))
)

# Step 3: Group by host-pair and aggregate everything into arrays
# collect_list() = pandas agg(list) equivalent
# first() gets the first value in the group (for scalar fields like DestinationIP)
http_df = http_df.groupBy(GROUP_BY_COLS).agg(
    F.count(f_timestamp).alias('conn_count'),
    F.collect_list('delta_seconds').alias('deltas'),        # array of time deltas (seconds)
    F.collect_list(f_sent_bytes).alias('sent_bytes_list'),  # array of sent byte values
    F.collect_list(f_received_bytes).alias('recv_bytes_list'),
    F.first(f_dst_ip).alias(f_dst_ip),
    F.first(f_dst_port).alias(f_dst_port),
)

# Remove null delta at the start of each group (the first row has no predecessor)
# array_compact() removes nulls from the array
http_df = http_df.withColumn(
    'deltas',
    F.array_compact(F.col('deltas'))
)

print(f"Grouped pairs: {http_df.count():,}")
http_df.show(3, truncate=True)

## Cell 5 — Filter short sessions

**What's happening:**  
Same logic as the original — drop any host-pair with fewer than `MIN_CONN_COUNT` connections. There's no statistical meaning in 5 connections.

This is a cheap filter that dramatically reduces the number of rows the expensive UDFs in later cells have to process.

In [ ]:
http_df = http_df.filter(F.col('conn_count') >= MIN_CONN_COUNT)

print(f"Pairs after minimum connection filter: {http_df.count():,}")

## Cell 6 — Statistical variables (time delta + data size)

**What's happening:**  

This is where the ACCD algorithm's core statistics are calculated. We're computing the same values as the original — 15th, 30th, 45th percentiles and MAD (Median Absolute Deviation) — but using **pandas UDFs**.

A pandas UDF (`@F.udf`) is a Python function that PySpark can run on each row. It takes a column value (here, an array), runs Python/numpy logic on it, and returns a scalar. This is the bridge between PySpark's distributed model and numpy's statistical functions.

**Why these percentiles?**  
The algorithm focuses on the **lower 45% of the time delta distribution** (the active/interactive phase). Higher deltas represent the beacon sleeping — those are exactly what attackers manipulate to evade detection, so we deliberately ignore them.

In [ ]:
# ── UDF definitions ───────────────────────────────────────────────────────────
# Each UDF takes an array column, runs numpy on it, returns a float.
# returnType tells Spark what Python type to expect back.

@F.udf(returnType=DoubleType())
def udf_percentile_15(arr):
    """15th percentile of an array"""
    a = [x for x in arr if x is not None]
    return float(np.percentile(a, 15)) if a else None

@F.udf(returnType=DoubleType())
def udf_percentile_30(arr):
    """30th percentile — used as the reference midpoint for MAD normalisation"""
    a = [x for x in arr if x is not None]
    return float(np.percentile(a, 30)) if a else None

@F.udf(returnType=DoubleType())
def udf_percentile_45(arr):
    """45th percentile — upper bound of the 'active phase' window"""
    a = [x for x in arr if x is not None]
    return float(np.percentile(a, 45)) if a else None

@F.udf(returnType=DoubleType())
def udf_madm_30(arr):
    """
    Median Absolute Deviation at the 30th percentile.
    MAD = median(|x - median(x)|) — measures dispersion around the median.
    Dividing by the median (tsMid) later gives a normalised jitter value.
    """
    a = np.array([x for x in arr if x is not None])
    if len(a) == 0:
        return None
    return float(np.percentile(np.absolute(a - np.median(a)), 30))

@F.udf(returnType=DoubleType())
def udf_max(arr):
    """Maximum value in an array"""
    a = [x for x in arr if x is not None]
    return float(max(a)) if a else None

# ── Apply UDFs to time delta arrays ──────────────────────────────────────────
http_df = (
    http_df
    .withColumn('tsLow',  udf_percentile_15('deltas'))  # 15th %ile of time deltas
    .withColumn('tsMid',  udf_percentile_30('deltas'))  # 30th %ile — jitter reference
    .withColumn('tsHigh', udf_percentile_45('deltas'))  # 45th %ile — active phase ceiling
    .withColumn('tsMadm', udf_madm_30('deltas'))        # Dispersion within active phase
)

# ── Apply UDFs to data size arrays ────────────────────────────────────────────
http_df = (
    http_df
    .withColumn('dsLow',         udf_percentile_15('sent_bytes_list'))
    .withColumn('dsMid',         udf_percentile_30('sent_bytes_list'))
    .withColumn('dsHigh',        udf_percentile_45('sent_bytes_list'))
    .withColumn('dsMadm',        udf_madm_30('sent_bytes_list'))
    .withColumn('dsResponseMax', udf_max('recv_bytes_list'))  # Max received bytes (C2 task output)
)

print("Statistical variables calculated")
http_df.select(
    f_src_ip, f_dst_host, 'conn_count',
    'tsLow', 'tsMid', 'tsHigh', 'tsMadm',
    'dsLow', 'dsMid', 'dsHigh', 'dsMadm', 'dsResponseMax'
).show(5)

## Cell 7 — Interactivity duration and connection count

**What's happening:**  
This is a critical FP filter from the original algorithm. It asks: within the active phase (deltas ≤ tsHigh), how long did the session last and how many connections occurred?

A legitimate beacon must have:
- At least `MIN_INTERACTIVITY_DURATION` seconds of active-phase traffic (default: 1 hour)
- At least `MIN_INTERACTIVITY_CONN` connections in that phase (default: 24)

This eliminates short bursts that score well statistically but aren't plausible as sustained C2.

The `0.2` second substitution for zero-delta connections handles the case where a beacon's sleep is set to 0 — the actual network round-trip means it can't truly be 0, but the log resolution rounds it down.

In [ ]:
@F.udf(returnType=DoubleType())
def udf_interactivity_duration(deltas, ts_high):
    """
    Sum of deltas that fall within the 'active phase' (delta <= tsHigh).
    Deltas of exactly 0 are treated as 0.2s (log resolution artifact).
    Returns the total seconds of the interactive beaconing window.
    """
    if not deltas or ts_high is None:
        return 0.0
    active = [d for d in deltas if d is not None and d <= ts_high]
    return float(sum(d if d > 0 else 0.2 for d in active))

@F.udf(returnType=IntegerType())
def udf_interactivity_conn_count(deltas, ts_high):
    """
    Count of connections within the active phase (delta <= tsHigh).
    """
    if not deltas or ts_high is None:
        return 0
    return int(sum(1 for d in deltas if d is not None and d <= ts_high))

http_df = (
    http_df
    .withColumn('interactivity_duration',
        udf_interactivity_duration(F.col('deltas'), F.col('tsHigh')))
    .withColumn('interactivity_conn_count',
        udf_interactivity_conn_count(F.col('deltas'), F.col('tsHigh')))
)

# Apply interactivity filters
http_df = http_df.filter(
    (F.col('interactivity_conn_count') >= MIN_INTERACTIVITY_CONN) &
    (F.col('interactivity_duration') >= MIN_INTERACTIVITY_DURATION)
)

print(f"Pairs after interactivity filter: {http_df.count():,}")

## Cell 8 — Execution bytes filter

**What's happening:**  
In a real attack, the C2 operator sends commands and tools *to* the beacon. The beacon's response (received bytes) should therefore include some large values — tool output, file transfers, etc.

This UDF finds the 10th-largest received-bytes value. If that value is below `MIN_EXECUTION_BYTES` (20KB), we consider the session insufficiently interactive to be plausible C2 — it might just be heartbeat-only traffic.

The `-MIN_EXECUTION_COUNT` index means: "give me the value at position [10th from the end] in the sorted list" — i.e. there must be at least 10 connections with responses larger than this.

In [ ]:
@F.udf(returnType=LongType())
def udf_min_execution_bytes(recv_bytes, min_count):
    """
    Returns the Nth-largest received-bytes value (where N = min_count).
    If fewer than min_count values exist, returns 0.
    """
    clean = sorted([x for x in recv_bytes if x is not None])
    if len(clean) < min_count:
        return 0
    return int(clean[-min_count])

http_df = http_df.withColumn(
    'min_execution_bytes',
    udf_min_execution_bytes(F.col('recv_bytes_list'), F.lit(MIN_EXECUTION_COUNT))
)

http_df = http_df.filter(F.col('min_execution_bytes') > MIN_EXECUTION_BYTES)

print(f"Pairs after execution bytes filter: {http_df.count():,}")

## Cell 9 — Score calculation

**What's happening:**  
The scoring logic is identical to the original. Two scores are calculated:

**`tsScore` (time delta score):**  
Measures how consistent the beacon's call-home interval is. `tsMadm / tsMid` gives normalised jitter. If jitter < 55%, score = 1 (very consistent). Above that, score degrades linearly.

**`dsScore` (data size score):**  
Same idea for sent bytes. The additional `dsResponseMax < 20000` penalty catches sessions where the beacon never received any substantial data — less likely to be active C2.

**`Score`:**  
Simple average of both. Range 0–1, where 1 = textbook beacon.

In [ ]:
@F.udf(returnType=DoubleType())
def udf_ts_score(ts_madm, ts_mid, jitter_threshold=55.0):
    """
    Time delta score.
    If normalised jitter (MAD/median * 100) is below threshold: score = 1.
    Otherwise: score degrades as jitter increases.
    tsMid == 0 means instantaneous repeated connections — treat as perfect beacon.
    """
    if ts_madm is None or ts_mid is None:
        return 0.0
    if ts_mid > 0:
        jitter_pct = (ts_madm / ts_mid) * 100
        if jitter_pct < jitter_threshold:
            return 1.0
        else:
            return 1.0 - ((ts_madm / ts_mid) * 0.4)
    else:
        return 1.0  # Zero median = perfectly tight timing

@F.udf(returnType=DoubleType())
def udf_ds_score(ds_madm, ds_mid, ds_response_max, jitter_threshold=25.0):
    """
    Data size score.
    Same jitter logic as ts_score but with a tighter threshold (25% vs 55%).
    Penalty of -0.3 if max received bytes < 20KB (no meaningful C2 tasking observed).
    """
    if ds_madm is None or ds_mid is None:
        return 0.0
    if ds_mid > 0:
        jitter_pct = (ds_madm / ds_mid) * 100
        if jitter_pct < jitter_threshold:
            score = 1.0
        else:
            score = 1.0 - ((ds_madm / ds_mid) * 0.4)
    else:
        score = 1.0
    if ds_response_max is not None and ds_response_max < 20000:
        score -= 0.3
    return score

http_df = (
    http_df
    .withColumn('tsScore', udf_ts_score(F.col('tsMadm'), F.col('tsMid')))
    .withColumn('dsScore', udf_ds_score(F.col('dsMadm'), F.col('dsMid'), F.col('dsResponseMax')))
)

# Final score = average of time delta score and data size score
http_df = http_df.withColumn('Score', (F.col('tsScore') + F.col('dsScore')) / 2.0)

# Filter to high-confidence detections
http_df = http_df.filter(F.col('Score') > MIN_SCORE)

print(f"High-score detections (Score > {MIN_SCORE}): {http_df.count():,}")

## Cell 10 — Destination prevalence enrichment

**What's happening:**  
Prevalence = how many distinct source IPs are talking to the same destination host (among our high-score detections).

High prevalence means "lots of machines beaconing to the same host" — which is actually *less* suspicious (could be a CDN, analytics endpoint, etc.) or could indicate widespread compromise. Low prevalence (1–4 machines) is more interesting — it suggests an isolated infected host.

This is done *after* score filtering intentionally — you don't want benign high-volume traffic inflating the prevalence count and masking true positives.

In PySpark this is a simple `groupBy` + `countDistinct` + `join` back to the main DataFrame.

In [ ]:
# Count distinct source IPs per destination host (among scored results only)
prevalence_df = (
    http_df
    .groupBy(f_dst_host)
    .agg(F.countDistinct(f_src_ip).alias('destination_prevalence'))
)

# Join prevalence back onto main results
http_df = http_df.join(prevalence_df, on=f_dst_host, how='left')

print("Prevalence enrichment complete")

## Cell 11 — Final results

**What's happening:**  
This is the only cell that calls `.toPandas()` — pulling results from the distributed Spark cluster into local memory. By this point the dataset is small (only high-score, low-prevalence detections) so this is safe.

We then apply the prevalence filter and display the output sorted by score descending.

In [ ]:
# Columns to display — matches the original notebook
display_cols = [
    'Score', 'tsScore', 'dsScore', 'conn_count',
    'min_execution_bytes', 'dsResponseMax', 'destination_prevalence',
    f_src_ip, f_dst_ip, f_dst_host, f_http_method, f_dst_port,
    'interactivity_duration', 'interactivity_conn_count', 'deltas'
]

# Pull to pandas for display (small result set by this point)
results_pd = (
    http_df
    .filter(F.col('destination_prevalence') < MAX_PREVALENCE)
    .select(display_cols)
    .toPandas()
    .sort_values('Score', ascending=False)
    .reset_index(drop=True)
)

print(f"Final detections: {len(results_pd)}")
results_pd

## Cell 12 (Optional) — Write results back to the data lake

**What's happening:**  
This writes the scored results as a custom table in the data lake tier. From there you can:
- Promote it to the analytics tier and create a Sentinel analytics rule against it
- Schedule this notebook as a job to run daily
- Join it with TI data in a follow-up notebook

Uncomment when you're ready to operationalise.

In [ ]:
# OUTPUT_TABLE = "ACCD_BeaconingDetections"

# Convert results back to a Spark DataFrame for write
# (drop the 'deltas' array column — it's large and not needed in the output table)
# output_df = spark.createDataFrame(results_pd.drop(columns=['deltas']))

# Write to the data lake tier
# data_provider.save_as_table(output_df, OUTPUT_TABLE, WORKSPACE_NAME)

# print(f"Results written to data lake table: {OUTPUT_TABLE}")